In [1]:
import Pkg
Pkg.activate(".")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/notebooks`


In [8]:
using StaticArrays

In [11]:
struct Points
    field::SVector{2, Float64}
    source::SVector{2, Float64}
end


function Points(field::NTuple{2, <:Real}, source::NTuple{2, <:Real})
    return Points(SVector{2, Float64}(field), SVector{2, Float64}(source))
end


function Points(field::AbstractArray{<:Real, 1}, source::AbstractArray{<:Real, 1})
    return Points(SVector{2, Float64}(field), SVector{2, Float64}(source))
end

Points

In [12]:
Points((1, 2), (3, 4))

Points([1.0, 2.0], [3.0, 4.0])

In [13]:
Points([1, 2], [3, 4])

Points([1.0, 2.0], [3.0, 4.0])

In [ ]:
abstract type AbstractPointsParameters


struct PointsHessianParameters <: AbstractPointsParameters
    v::NTuple{3, Float64}
    r::NTuple{3, Float64}
    R::Float64
    A::Float64
    X::Float64
    B::NTuple{3, Float64}
    V₃::Float64

    PointsParameters(wave::WaveParameters, field_point::NTuple{2, Float64}, source_point::NTuple{2, Float64})
        # Dimensional parameters
        h = wave.h
        K = wave.K
    
        x, z = field_point
        ξ, ζ = source_point
    
        R̄ = x - ξ
        R = abs(R̄)
        R² = R^2
        ∇R = sign(R̄)
    
        v̄₁ = z - ζ
        v₁ = abs(v̄₁)
        v₃ = -z - ζ
        v₂ = 2h - v₃
    
        v₁² = v₁^2
        v₂² = v₂^2
        v₃² = v₃^2
    
        r₁² = R² + v₁²
        r₁ = √r₁²
        r₁³ = r₁² * r₁
        r₁ˣ = R̄ / r₁
        r₁ᶻ = v̄₁ / r₁
        ∇r₁ = SVector{2, Float64}(r₁ˣ, r₁ᶻ)
        Hr₁ˣˣ = v₁² / r₁³
        Hr₁ˣᶻ = -R̄ * v̄₁ / r₁³
        Hr₁ᶻᶻ = R² / r₁³
        Hr₁ = SMatrix{2, 2, Float64}([Hr₁ˣˣ Hr₁ˣᶻ; Hr₁ˣᶻ Hr₁ᶻᶻ])
    
        r₂² = R² + v₂²
        r₂ = √r₂²
        r₂³ = r₂² * r₂
        r₂ˣ = R̄ / r₂
        r₂ᶻ = v₂ / r₂
        ∇r₂ = SVector{2, Float64}(r₂ˣ, r₂ᶻ)
        Hr₂ˣˣ = v₂² / r₂³
        Hr₂ˣᶻ = -R̄ * v₂ / r₂³
        Hr₂ᶻᶻ = R² / r₂³
        Hr₂ = SMatrix{2, 2, Float64}([Hr₂ˣˣ Hr₂ˣᶻ; Hr₂ˣᶻ Hr₂ᶻᶻ])
    
        r₃² = R² + v₃²
        r₃ = √r₃²
        r₃³ = r₃² * r₃
        r₃ˣ = R̄ / r₃
        r₃ᶻ = -v₃ / r₃
        ∇r₃ = SVector{2, Float64}(r₃ˣ, r₃ᶻ)
        Hr₃ˣˣ = v₃² / r₃³
        Hr₃ˣᶻ = R̄ * v₃ / r₃³
        Hr₃ᶻᶻ = R² / r₃³
        Hr₃ = SMatrix{2, 2, Float64}([Hr₃ˣˣ Hr₃ˣᶻ; Hr₃ˣᶻ Hr₃ᶻᶻ])
    
        # Dimensionless parameters
        H = K * h
    
        A = R / h
        ∇A = ∇R / h
    
        B₁ = v₁ / h
        B₂ = v₂ / h
    
        ∇B₁ = sign(v̄₁) / h
        ∇B₂ = 1 / h
    
        V₃ = K * v₃
        X = K * R
        Z = V₃ - im*X
        ∇Z = SVector{2, ComplexF64}(-im*K*∇R, -K)
    
        # Rankine sources
        r = (r₁, r₂, r₃)
        ∇r = (∇r₁, ∇r₂, ∇r₃)
        Hr = (Hr₁, Hr₂, Hr₃)
    
        # Integrals L₁ and L₂
        u₁ = SVector{2,Float64}(A, B₁)
        u₂ = SVector{2,Float64}(A, B₂)
    
        ∇u₁ = SVector{2,Float64}(∇A, ∇B₁)
        ∇u₂ = SVector{2,Float64}(∇A, ∇B₂)
    
        new(R, ∇R, v, ∇v, r, ∇r, Hr)
    end
end

In [3]:
s = (3.3, 1.2, 0.5)

(3.3, 1.2, 0.5)

In [5]:
typeof(s) <: NTuple{3, Float64}

true

In [6]:
s isa NTuple{3, Float64}

true

In [2]:
?NTuple

search: NTuple Tuple tuple ntuple NamedTuple @NamedTuple Type



```julia
NTuple{N, T}
```

A compact way of representing the type for a tuple of length `N` where all elements are of type `T`.

# Examples

```jldoctest
julia> isa((1, 2, 3, 4, 5, 6), NTuple{6, Int})
true
```

See also [`ntuple`](@ref).
